# Spectrogram Analysis and Recommendation System
This notebook analyzes audio previews using `scipy.signal.spectrogram`, extracts frequency band features, and builds a recommendation system based on cosine similarity.

In [1]:
%pip install scipy numpy pandas scikit-learn librosa tqdm

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import glob
import numpy as np
import pandas as pd
import librosa
from scipy import signal
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')


In [3]:
NUM_SONGS_TO_PROCESS = 5000
PREVIEWS_DIR = 'data/previews'
DATASET_PATH = 'spotify_5000.parquet'
OUTPUT_CSV_PATH = 'spectrogram_features.csv'


In [4]:
def get_spectrogram_features(file_path):
    try:
        # Load audio file. librosa automatically delegates to audioread for mp3
        y, sr = librosa.load(file_path, sr=22050, mono=True)
        # Generate spectrogram using scipy
        f, t, Sxx = signal.spectrogram(y, fs=sr)
        # Return mean and std energy per frequency bin as a feature vector
        mean_energy = np.mean(Sxx, axis=1)
        std_energy = np.std(Sxx, axis=1)
        return np.concatenate([mean_energy, std_energy])
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return None


In [5]:
# Get list of mp3 files
preview_files = glob.glob(os.path.join(PREVIEWS_DIR, '*.mp3'))
preview_files = preview_files[:NUM_SONGS_TO_PROCESS]

features_list = []
track_ids = []

print(f"Processing {len(preview_files)} files...")
for file_path in tqdm(preview_files):
    track_id = os.path.basename(file_path).replace('.mp3', '')
    features = get_spectrogram_features(file_path)
    if features is not None:
        features_list.append(features)
        track_ids.append(track_id)

if len(features_list) > 0:
    num_features = len(features_list[0])
    feature_cols = [f'spec_feat_{i}' for i in range(num_features)]
    df_features = pd.DataFrame(features_list, columns=feature_cols)
    df_features['track_id'] = track_ids
else:
    df_features = pd.DataFrame(columns=['track_id'])
    feature_cols = []
    
print(f"Successfully processed {len(df_features)} tracks.")
df_features.head()


Processing 1000 files...


100%|██████████| 1000/1000 [00:36<00:00, 27.04it/s]

Successfully processed 1000 tracks.


,spec_feat_0,spec_feat_1,spec_feat_2,spec_feat_3,spec_feat_4,spec_feat_5,spec_feat_6,spec_feat_7,spec_feat_8,spec_feat_9,...,spec_feat_249,spec_feat_250,spec_feat_251,spec_feat_252,spec_feat_253,spec_feat_254,spec_feat_255,spec_feat_256,spec_feat_257,track_id
0,0.000002,0.000007,0.000010,0.000059,0.000134,0.000077,0.000015,0.000020,0.000036,0.000028,...,1.083484e-07,7.316153e-08,1.206669e-07,1.943041e-08,4.944601e-09,1.043337e-09,4.805296e-10,1.969282e-10,6.203713e-11,7DmZ1e9hh9CUX0PZzZA25C
1,0.000001,0.000006,0.000024,0.000041,0.000040,0.000016,0.000016,0.000033,0.000019,0.000010,...,1.231868e-07,6.185755e-08,3.468129e-08,1.099820e-08,2.070984e-09,6.000422e-10,1.817871e-10,1.167456e-10,5.023744e-11,5wHf0cE3iNhw76HgsBNC3y
2,0.000006,0.000123,0.000220,0.000139,0.000092,0.000111,0.000091,0.000068,0.000040,0.000043,...,5.417165e-07,3.302822e-07,2.251723e-07,8.848315e-08,2.684272e-08,5.920035e-09,1.982534e-09,1.088697e-09,4.826725e-10,5kvsEeN7tv7iqbR4P4do8e
3,0.000001,0.000047,0.000036,0.000031,0.000021,0.000012,0.000008,0.000009,0.000005,0.000005,...,1.664158e-07,1.263652e-07,6.927179e-08,2.485055e-08,6.557859e-09,1.731662e-09,5.277698e-10,2.841426e-10,1.244440e-10,0LAztf40276x1QkIUnYjxo
4,0.000003,0.000092,0.000098,0.000087,0.000036,0.000026,0.000022,0.000039,0.000014,0.000015,...,2.661957e-07,1.340966e-07,8.604778e-08,2.321030e-08,6.361090e-09,1.670836e-09,6.775892e-10,4.353425e-10,2.013105e-10,37fi1q24yJTmDcP9FFqq4T


In [6]:
# Save spectrogram features to CSV
if not df_features.empty:
    df_features.to_csv(OUTPUT_CSV_PATH, index=False)
    print(f"Saved spectrogram features to {OUTPUT_CSV_PATH}")


Saved spectrogram features to /home/build/martin/spotify/spectrogram_features.csv


In [7]:
# Load Main Dataset
df_main = pd.read_parquet(DATASET_PATH)

# Determine the track ID column in main dataset
join_col = 'track_id' if 'track_id' in df_main.columns else 'id'

# Merge features with main dataset
df_merged = pd.merge(df_main, df_features, left_on=join_col, right_on='track_id', how='inner')
print(f"Merged dataset shape: {df_merged.shape}")
df_merged.head()


Merged dataset shape: (928, 299)


,track_id,album_name,track_name,popularity,explicit,danceability,energy,key,loudness,mode,...,spec_feat_248,spec_feat_249,spec_feat_250,spec_feat_251,spec_feat_252,spec_feat_253,spec_feat_254,spec_feat_255,spec_feat_256,spec_feat_257
0,5Z8O4RFAmjA13jrOYIgyXA,Oh What A Love,Oh What A Love,46,False,0.457,0.064,6,-19.928,1,...,5.625330e-09,4.376840e-09,3.728543e-09,1.572201e-09,5.714405e-10,1.210977e-10,3.365253e-11,1.389587e-11,6.399252e-12,2.387912e-12
1,5s6ppi6NDeK6FXlX8XCajM,Dancing with the Curse,8 Track,28,False,0.408,0.974,1,-5.501,1,...,7.809188e-08,7.179657e-08,4.740392e-08,3.993483e-08,1.399222e-08,3.184186e-09,8.735479e-10,2.877863e-10,1.507661e-10,6.694591e-11
2,0qAMjeQFyd1qD0LDiV8gWp,Eye To The Telescope,Black Horse And The Cherry Tree,67,False,0.748,0.786,4,-7.788,0,...,3.230133e-07,3.012684e-07,1.518282e-07,7.473494e-08,2.897971e-08,1.063141e-08,2.300325e-09,8.023061e-10,3.887721e-10,1.577420e-10
3,2TPCINjlzZ9Hbp0tCRQC2b,BaianaSystem,O Carnaval Quem É Que Faz?,29,False,0.603,0.849,11,-8.917,0,...,2.439933e-07,1.042707e-07,6.514509e-08,5.817794e-08,2.654097e-08,5.064244e-09,1.152609e-09,4.782823e-10,2.849385e-10,1.285145e-10
4,4l5OJLb3AsvfEyeAYAJHpe,La Makinita,Venga Mi Vida,26,False,0.698,0.542,9,-5.923,0,...,1.791143e-07,1.229477e-07,8.835225e-08,8.993310e-08,3.789686e-08,1.381628e-08,2.247431e-09,7.217344e-10,3.488687e-10,1.393482e-10


In [10]:
# Recommendation System based on spectrogram features
def get_recommendations(target_track_id, df, features_columns, top_n=5):
    # Determine ID column in merged df
    id_col = 'track_id' if 'track_id' in df.columns else 'id'
    
    if target_track_id not in df[id_col].values:
        return "Track ID not found in the merged dataset."
    
    # Extract feature matrix
    feature_matrix = df[features_columns].values
    
    # Get target track features
    target_idx = df.index[df[id_col] == target_track_id].tolist()[0]
    target_features = feature_matrix[target_idx].reshape(1, -1)
    
    # Calculate cosine similarity
    similarities = cosine_similarity(target_features, feature_matrix)[0]
    
    # Add similarities to a temp dataframe
    df_sim = df.copy()
    df_sim['similarity'] = similarities
    
    # Sort and get top N (excluding the target track itself)
    recommendations = df_sim[df_sim[id_col] != target_track_id].sort_values(by='similarity', ascending=False).head(top_n)
    
    # Select columns to display
    display_cols = [id_col, 'similarity']
    for col in ['name', 'track_name', 'artists', 'artist_name']:
         if col in df_sim.columns:
             display_cols.append(col)
             
    return recommendations[display_cols]


# Example: Get recommendations for the first valid track in our processed subset
if not df_merged.empty:
    id_col = 'track_id' if 'track_id' in df_merged.columns else 'id'
    sample_track = df_merged.iloc[0][id_col]
    sample_track_name = df_merged.iloc[0].get('track_name', df_merged.iloc[0].get('name', 'Unknown Track'))
    print(f"Recommendations for Track ID: {sample_track} - Track Name: {sample_track_name}")
    display(get_recommendations(sample_track, df_merged, feature_cols))


Recommendations for Track ID: 5Z8O4RFAmjA13jrOYIgyXA - Track Name: Oh What A Love


,track_id,similarity,name,track_name,artists
823,5Infc095QF0BDwCF9rMyxD,0.989569,Pareshaan,Pareshaan,Suzonn
300,7ovNetXenusbaOmWanOWJC,0.973721,The Morning After,The Morning After,Kid Loco
675,0LAztf40276x1QkIUnYjxo,0.965433,Na Na Ni Na Não,Na Na Ni Na Não,Di Propósito
411,6SB0NzwnyO71TI6z8pMIiE,0.964428,Savannah's Assault,Savannah's Assault,Phobia
831,2h3NyiQQbUHavbvtrkeevt,0.964310,Hanna kommer hem,Hanna kommer hem,Lasse Stefanz
